# 03 — LSTM and TCN

**LSTM.** A recurrent architecture (two stacked LSTM layers + dropout +
dense output) capable of learning nonlinear, longer-range temporal
dependencies that a linear model cannot, at the cost of needing more data
and compute per prediction and an opaque internal representation.

**TCN (Temporal Convolutional Network).** A new model not present in the
original codebase, added here as a second nonlinear sequence learner for
comparison: a stack of dilated causal 1D convolutions with residual
connections. Architecture (confirmed): **6 residual blocks**, dilations
`1, 2, 4, 8, 16, 32`, kernel size 3, 32 filters/block, causal padding,
dropout 0.2 (matching LSTM's dropout), giving a receptive field of ~253
steps, comfortably covering the 144-step (1-day) input window used
elsewhere in this project. Each block: `Conv1D -> ReLU -> Dropout -> Conv1D
-> ReLU -> Dropout -> residual add` (with a 1x1 conv projection on the
shortcut when channel counts differ), followed by `GlobalAveragePooling1D`
+ `Dense(1)` head. Unlike LSTM, a TCN processes the whole input window in
parallel (no recurrence), which is typically much faster to train.

Both models consume the same input as each other: a sliding window of
`sequence_length` scaled `internet` values (single feature), and are
evaluated with the same walk-forward, ground-truth-conditioned one-step-ahead
protocol used in `02_sarima.ipynb`.

**Structure:** model classes → tuning for each model, with a markdown cell
after each round → final walk-forward evaluation on all 3 target squares →
predictions/metrics saved to `results/` for `04_model_comparison.ipynb`.

In [1]:
import ast
import os
import pickle
import sys
import time

import numpy as np
import pandas as pd
import yaml

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, ".")

import common
from common import EVAL_WEEK_START, EVAL_WEEK_END, TRAIN_START

common.set_seed()
os.makedirs("results", exist_ok=True)

VAL_DAYS = 3
LSTM_GRID = {"sequence_length": [72, 144], "units": [[32], [64, 32]], "learning_rate": [0.001, 0.0005]}
TCN_GRID = {"sequence_length": [72, 144], "filters": [16, 32], "learning_rate": [0.001, 0.0005]}
TCN_DILATIONS = (1, 2, 4, 8, 16, 32)
TCN_KERNEL_SIZE = 3
FINAL_BATCH_SIZE = 64
FINAL_EPOCHS = 40
FINAL_EARLY_STOPPING_PATIENCE = 5

meta = common.load_target_squares_meta()
print("Top-3 squares:", meta["top3_squares"])

Top-3 squares: [5161, 5059, 5259]


## 1. Shared sequence helper

In [2]:
def create_sequences(data: np.ndarray, sequence_length: int):
    """Turn a 1D scaled series into overlapping (X, y) supervised sequences."""
    X, y = [], []
    for i in range(len(data) - sequence_length):
        X.append(data[i : i + sequence_length])
        y.append(data[i + sequence_length])
    return np.array(X), np.array(y)

## 2. Model classes

In [3]:
class LSTMModel:
    """Wrapper around a Keras Sequential LSTM for a consistent fit/predict API."""

    def __init__(self, sequence_length=24, units=(64, 32), dropout=0.2, learning_rate=0.001):
        self.sequence_length = sequence_length
        self.units = list(units)
        self.dropout = dropout
        self.learning_rate = learning_rate
        self.model = None

    def _build(self):
        from tensorflow.keras.layers import LSTM, Dense, Dropout
        from tensorflow.keras.models import Sequential
        from tensorflow.keras.optimizers import Adam

        model = Sequential()
        for i, n_units in enumerate(self.units):
            return_sequences = i < len(self.units) - 1
            if i == 0:
                model.add(LSTM(n_units, return_sequences=return_sequences,
                                input_shape=(self.sequence_length, 1)))
            else:
                model.add(LSTM(n_units, return_sequences=return_sequences))
            model.add(Dropout(self.dropout))
        model.add(Dense(1))
        model.compile(optimizer=Adam(learning_rate=self.learning_rate), loss="mse")
        self.model = model

    def fit(self, X_train, y_train, X_val=None, y_val=None, batch_size=32, epochs=50, early_stopping_patience=5):
        from tensorflow.keras.callbacks import EarlyStopping
        if self.model is None:
            self._build()
        X_train = X_train.reshape((*X_train.shape, 1))
        validation_data, callbacks = None, []
        if X_val is not None and y_val is not None and len(X_val) > 0:
            X_val = X_val.reshape((*X_val.shape, 1))
            validation_data = (X_val, y_val)
            callbacks.append(EarlyStopping(monitor="val_loss", patience=early_stopping_patience,
                                            restore_best_weights=True))
        self.model.fit(X_train, y_train, validation_data=validation_data, batch_size=batch_size,
                        epochs=epochs, callbacks=callbacks, verbose=0)
        return self

    def predict(self, X):
        if self.model is None:
            raise RuntimeError("Call fit() before predict().")
        X = X.reshape((*X.shape, 1))
        return self.model.predict(X, verbose=0).flatten()

In [4]:
class TCNModel:
    """Dilated causal Conv1D (TCN) wrapper, matching LSTMModel's fit/predict API."""

    def __init__(self, sequence_length=144, filters=32, kernel_size=TCN_KERNEL_SIZE,
                 dilations=TCN_DILATIONS, dropout=0.2, learning_rate=0.001):
        self.sequence_length = sequence_length
        self.filters = filters
        self.kernel_size = kernel_size
        self.dilations = list(dilations)
        self.dropout = dropout
        self.learning_rate = learning_rate
        self.model = None

    def _residual_block(self, x, dilation):
        from tensorflow.keras.layers import Add, Conv1D, Dropout, ReLU
        in_channels = x.shape[-1]
        h = Conv1D(self.filters, self.kernel_size, dilation_rate=dilation, padding="causal")(x)
        h = ReLU()(h)
        h = Dropout(self.dropout)(h)
        h = Conv1D(self.filters, self.kernel_size, dilation_rate=dilation, padding="causal")(h)
        h = ReLU()(h)
        h = Dropout(self.dropout)(h)
        shortcut = x if in_channels == self.filters else Conv1D(self.filters, 1, padding="same")(x)
        return Add()([shortcut, h])

    def _build(self):
        from tensorflow.keras.layers import Dense, GlobalAveragePooling1D, Input
        from tensorflow.keras.models import Model
        from tensorflow.keras.optimizers import Adam

        inputs = Input(shape=(self.sequence_length, 1))
        x = inputs
        for dilation in self.dilations:
            x = self._residual_block(x, dilation)
        x = GlobalAveragePooling1D()(x)
        outputs = Dense(1)(x)
        model = Model(inputs, outputs)
        model.compile(optimizer=Adam(learning_rate=self.learning_rate), loss="mse")
        self.model = model

    def fit(self, X_train, y_train, X_val=None, y_val=None, batch_size=32, epochs=50, early_stopping_patience=5):
        from tensorflow.keras.callbacks import EarlyStopping
        if self.model is None:
            self._build()
        X_train = X_train.reshape((*X_train.shape, 1))
        validation_data, callbacks = None, []
        if X_val is not None and y_val is not None and len(X_val) > 0:
            X_val = X_val.reshape((*X_val.shape, 1))
            validation_data = (X_val, y_val)
            callbacks.append(EarlyStopping(monitor="val_loss", patience=early_stopping_patience,
                                            restore_best_weights=True))
        self.model.fit(X_train, y_train, validation_data=validation_data, batch_size=batch_size,
                        epochs=epochs, callbacks=callbacks, verbose=0)
        return self

    def predict(self, X):
        if self.model is None:
            raise RuntimeError("Call fit() before predict().")
        X = X.reshape((*X.shape, 1))
        return self.model.predict(X, verbose=0).flatten()

## 3. LSTM tuning

Grid search on the top-traffic square, 15 tuning epochs with early stopping
(patience 3) to keep the grid tractable; the winning configuration is
retrained for longer (`FINAL_EPOCHS`) in the final evaluation below. Rounds
are grouped by `sequence_length`, sweeping `units` x `learning_rate` within
each round.

In [5]:
tune_square = meta["top3_squares"][0]
series = common.load_square_series(tune_square)
eval_start = pd.Timestamp(EVAL_WEEK_START)
train_end = eval_start - pd.Timedelta(days=VAL_DAYS)
train_full = series[series.index < train_end]
val_full = series[(series.index >= train_end) & (series.index < eval_start)]
print(f"Tuning on square {tune_square}: train={len(train_full)} pts, val={len(val_full)} pts")


def tune_one_lstm_round(seq_len, units_list, lr_list, train, val):
    rows = []
    for units in units_list:
        for lr in lr_list:
            t0 = time.perf_counter()
            common.set_seed()
            scaler = common.SeriesScaler("minmax")
            train_scaled = scaler.fit_transform(train).flatten()
            full_val_input = pd.concat([train.tail(seq_len), val])
            val_scaled = scaler.transform(full_val_input).flatten()

            X_train, y_train = create_sequences(train_scaled, seq_len)
            X_val, y_val = create_sequences(val_scaled, seq_len)

            model = LSTMModel(seq_len, units, dropout=0.2, learning_rate=lr)
            model.fit(X_train, y_train, batch_size=64, epochs=15, early_stopping_patience=3)
            preds = scaler.inverse_transform(model.predict(X_val))
            metrics = common.compute_metrics(val.values, preds)
            elapsed = time.perf_counter() - t0
            row = {"model": "LSTM", "sequence_length": seq_len, "units": str(units), "learning_rate": lr,
                   "rmse": metrics["RMSE"], "mae": metrics["MAE"], "mape": metrics["MAPE"], "elapsed_s": elapsed}
            rows.append(row)
            print(f"  seq={seq_len} units={units} lr={lr} -> RMSE={metrics['RMSE']:.2f} "
                  f"MAE={metrics['MAE']:.2f} MAPE={metrics['MAPE']:.2f} ({elapsed:.1f}s)")
    return rows

Tuning on square 5161: train=6054 pts, val=432 pts


In [6]:
print("=== LSTM Round 1: sequence_length=72 ===")
lstm_round1 = tune_one_lstm_round(72, LSTM_GRID["units"], LSTM_GRID["learning_rate"], train_full, val_full)
pd.DataFrame(lstm_round1)

=== LSTM Round 1: sequence_length=72 ===


/Users/manziivan453icloud.com/Downloads/Projects/IVAN/Formative1_Techinique1/venv/lib/python3.11/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


  seq=72 units=[32] lr=0.001 -> RMSE=200.53 MAE=138.96 MAPE=11.75 (10.3s)


/Users/manziivan453icloud.com/Downloads/Projects/IVAN/Formative1_Techinique1/venv/lib/python3.11/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


  seq=72 units=[32] lr=0.0005 -> RMSE=235.11 MAE=161.35 MAPE=13.74 (10.3s)


/Users/manziivan453icloud.com/Downloads/Projects/IVAN/Formative1_Techinique1/venv/lib/python3.11/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


  seq=72 units=[64, 32] lr=0.001 -> RMSE=220.24 MAE=155.48 MAPE=13.00 (39.3s)


/Users/manziivan453icloud.com/Downloads/Projects/IVAN/Formative1_Techinique1/venv/lib/python3.11/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


  seq=72 units=[64, 32] lr=0.0005 -> RMSE=229.63 MAE=161.56 MAPE=14.59 (38.9s)


,model,sequence_length,units,learning_rate,rmse,mae,mape,elapsed_s
0,LSTM,72,[32],0.0010,200.534225,138.960302,11.750472,10.309142
1,LSTM,72,[32],0.0005,235.112929,161.349214,13.739284,10.274084
2,LSTM,72,"[64, 32]",0.0010,220.240589,155.484907,13.002072,39.322098
3,LSTM,72,"[64, 32]",0.0005,229.625539,161.560453,14.587665,38.930514


**LSTM Round 1 result.** With a 72-step (12-hour) input window, the model
only ever sees half a day of history at once -- less than one full seasonal
cycle (144 steps = 1 day, per the EDA). **Next adjustment:** try the full
144-step window, which should let the network see a complete daily cycle in
a single input and is expected to help most for the smaller `[32]`-unit
configuration, which has the least capacity to compensate via depth.

In [7]:
print("=== LSTM Round 2: sequence_length=144 ===")
lstm_round2 = tune_one_lstm_round(144, LSTM_GRID["units"], LSTM_GRID["learning_rate"], train_full, val_full)
pd.DataFrame(lstm_round2)

=== LSTM Round 2: sequence_length=144 ===


/Users/manziivan453icloud.com/Downloads/Projects/IVAN/Formative1_Techinique1/venv/lib/python3.11/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


  seq=144 units=[32] lr=0.001 -> RMSE=285.87 MAE=198.83 MAPE=14.13 (19.3s)


/Users/manziivan453icloud.com/Downloads/Projects/IVAN/Formative1_Techinique1/venv/lib/python3.11/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


  seq=144 units=[32] lr=0.0005 -> RMSE=308.34 MAE=217.59 MAPE=17.44 (18.7s)


/Users/manziivan453icloud.com/Downloads/Projects/IVAN/Formative1_Techinique1/venv/lib/python3.11/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


  seq=144 units=[64, 32] lr=0.001 -> RMSE=228.32 MAE=166.43 MAPE=17.36 (77.3s)


/Users/manziivan453icloud.com/Downloads/Projects/IVAN/Formative1_Techinique1/venv/lib/python3.11/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


  seq=144 units=[64, 32] lr=0.0005 -> RMSE=291.15 MAE=196.42 MAPE=13.37 (75.6s)


,model,sequence_length,units,learning_rate,rmse,mae,mape,elapsed_s
0,LSTM,144,[32],0.0010,285.872303,198.833998,14.126033,19.285874
1,LSTM,144,[32],0.0005,308.337640,217.586856,17.442066,18.668880
2,LSTM,144,"[64, 32]",0.0010,228.318347,166.432635,17.357291,77.288314
3,LSTM,144,"[64, 32]",0.0005,291.146652,196.420487,13.372470,75.634597


**LSTM Round 2 result and decision.** Compare the two rounds' RMSE
columns above: the pattern determines whether the extra context from
`sequence_length=144` earns back its added compute cost. The best
combination across both rounds (lowest validation RMSE) is selected below
and used for the final evaluation.

In [8]:
lstm_tuning_df = pd.DataFrame(lstm_round1 + lstm_round2)
best_lstm_row = lstm_tuning_df.loc[lstm_tuning_df["rmse"].idxmin()]
best_lstm = {
    "sequence_length": int(best_lstm_row["sequence_length"]),
    "units": ast.literal_eval(best_lstm_row["units"]),
    "learning_rate": float(best_lstm_row["learning_rate"]),
}
print("Best LSTM config:", best_lstm, "RMSE:", best_lstm_row["rmse"])

Best LSTM config: {'sequence_length': 72, 'units': [32], 'learning_rate': 0.001} RMSE: 200.53422514889544


## 4. TCN tuning

Same protocol as LSTM (15 tuning epochs, early stopping patience 3),
grouped into rounds by `sequence_length`, sweeping `filters` x
`learning_rate` within each round. Architecture (block count, dilation
schedule, kernel size) is held fixed at the values decided above.

In [9]:
def tune_one_tcn_round(seq_len, filters_list, lr_list, train, val):
    rows = []
    for filters in filters_list:
        for lr in lr_list:
            t0 = time.perf_counter()
            common.set_seed()
            scaler = common.SeriesScaler("minmax")
            train_scaled = scaler.fit_transform(train).flatten()
            full_val_input = pd.concat([train.tail(seq_len), val])
            val_scaled = scaler.transform(full_val_input).flatten()

            X_train, y_train = create_sequences(train_scaled, seq_len)
            X_val, y_val = create_sequences(val_scaled, seq_len)

            model = TCNModel(seq_len, filters=filters, dropout=0.2, learning_rate=lr)
            model.fit(X_train, y_train, batch_size=64, epochs=15, early_stopping_patience=3)
            preds = scaler.inverse_transform(model.predict(X_val))
            metrics = common.compute_metrics(val.values, preds)
            elapsed = time.perf_counter() - t0
            row = {"model": "TCN", "sequence_length": seq_len, "filters": filters, "learning_rate": lr,
                   "rmse": metrics["RMSE"], "mae": metrics["MAE"], "mape": metrics["MAPE"], "elapsed_s": elapsed}
            rows.append(row)
            print(f"  seq={seq_len} filters={filters} lr={lr} -> RMSE={metrics['RMSE']:.2f} "
                  f"MAE={metrics['MAE']:.2f} MAPE={metrics['MAPE']:.2f} ({elapsed:.1f}s)")
    return rows

In [10]:
print("=== TCN Round 1: sequence_length=72 ===")
tcn_round1 = tune_one_tcn_round(72, TCN_GRID["filters"], TCN_GRID["learning_rate"], train_full, val_full)
pd.DataFrame(tcn_round1)

=== TCN Round 1: sequence_length=72 ===


  seq=72 filters=16 lr=0.001 -> RMSE=333.78 MAE=210.85 MAPE=16.27 (14.1s)


  seq=72 filters=16 lr=0.0005 -> RMSE=382.36 MAE=239.59 MAPE=16.30 (13.9s)


  seq=72 filters=32 lr=0.001 -> RMSE=275.29 MAE=216.72 MAPE=29.74 (24.6s)


  seq=72 filters=32 lr=0.0005 -> RMSE=316.20 MAE=256.45 MAPE=44.32 (26.4s)


,model,sequence_length,filters,learning_rate,rmse,mae,mape,elapsed_s
0,TCN,72,16,0.0010,333.775214,210.846654,16.270173,14.074822
1,TCN,72,16,0.0005,382.362626,239.590863,16.303744,13.928375
2,TCN,72,32,0.0010,275.290986,216.721995,29.744235,24.623475
3,TCN,72,32,0.0005,316.196379,256.447861,44.323151,26.363496


**TCN Round 1 result.** With dilations up to 32 and kernel size 3, the
receptive field (~253 steps) already exceeds a 72-step input, so the network
isn't receptive-field-starved here the way a shallower TCN would be --
any difference between `filters=16` and `filters=32` at this window length
reflects representational capacity, not missing context. **Next
adjustment:** try the full 144-step window to match LSTM's comparison
point and check whether the extra history helps a convolutional model the
way it helps the recurrent one.

In [11]:
print("=== TCN Round 2: sequence_length=144 ===")
tcn_round2 = tune_one_tcn_round(144, TCN_GRID["filters"], TCN_GRID["learning_rate"], train_full, val_full)
pd.DataFrame(tcn_round2)

=== TCN Round 2: sequence_length=144 ===


  seq=144 filters=16 lr=0.001 -> RMSE=289.13 MAE=203.27 MAPE=19.20 (24.7s)


  seq=144 filters=16 lr=0.0005 -> RMSE=326.99 MAE=211.68 MAPE=18.47 (27.8s)


  seq=144 filters=32 lr=0.001 -> RMSE=239.41 MAE=164.41 MAPE=13.97 (51.3s)


  seq=144 filters=32 lr=0.0005 -> RMSE=295.84 MAE=202.14 MAPE=14.54 (44.3s)


,model,sequence_length,filters,learning_rate,rmse,mae,mape,elapsed_s
0,TCN,144,16,0.0010,289.130869,203.274639,19.202203,24.661915
1,TCN,144,16,0.0005,326.991243,211.676983,18.467147,27.761199
2,TCN,144,32,0.0010,239.414840,164.411145,13.969988,51.312761
3,TCN,144,32,0.0005,295.839437,202.144466,14.538510,44.306612


**TCN Round 2 result and decision.** As with LSTM, compare both rounds'
RMSE to decide whether `sequence_length=144` earns its cost for a model
whose receptive field already covers 72 steps comfortably. The best
combination is selected below.

In [12]:
tcn_tuning_df = pd.DataFrame(tcn_round1 + tcn_round2)
best_tcn_row = tcn_tuning_df.loc[tcn_tuning_df["rmse"].idxmin()]
best_tcn = {
    "sequence_length": int(best_tcn_row["sequence_length"]),
    "filters": int(best_tcn_row["filters"]),
    "learning_rate": float(best_tcn_row["learning_rate"]),
}
print("Best TCN config:", best_tcn, "RMSE:", best_tcn_row["rmse"])

pd.concat([lstm_tuning_df, tcn_tuning_df], ignore_index=True).to_csv("results/tuning_results_lstm_tcn.csv", index=False)

best_params_path = "results/best_params.yaml"
best_params = {}
if os.path.exists(best_params_path):
    with open(best_params_path) as f:
        best_params = yaml.safe_load(f) or {}
best_params["lstm"] = {**best_lstm, "tuned_on_square": tune_square, "validation_window_days": VAL_DAYS}
best_params["tcn"] = {**best_tcn, "kernel_size": TCN_KERNEL_SIZE, "dilations": list(TCN_DILATIONS),
                       "tuned_on_square": tune_square, "validation_window_days": VAL_DAYS}
with open(best_params_path, "w") as f:
    yaml.safe_dump(best_params, f)
print(f"Saved best LSTM/TCN params to {best_params_path}")

Best TCN config: {'sequence_length': 144, 'filters': 32, 'learning_rate': 0.001} RMSE: 239.41484021955193
Saved best LSTM/TCN params to results/best_params.yaml


## 5. Final walk-forward evaluation on all 3 target squares

Same walk-forward one-step protocol as SARIMA, but ground-truth-conditioned
rather than append-based: at each step, both models predict from the *true*
preceding `sequence_length` values (teacher forcing), which is equivalent
in spirit to SARIMAX's `.append(refit=False)` state update -- both use only
real observed history, never the model's own prior predictions, to produce
each one-step-ahead forecast.

**Validation-gated early stopping in the final run.** The fit below carves
out the same `VAL_DAYS`-day slice from the end of the training window that
tuning uses (Sections 3/4), and passes it as real `X_val`/`y_val` so early
stopping is actually active here -- not just during tuning. The scaler is
fit on the reduced training window only (excluding the validation slice),
matching tuning's leakage-avoidance exactly; the *prediction* input context
for the eval week still uses the full training history's tail (all real
data up to `eval_start`), since once fitting is complete there is no reason
to withhold the most recent points from inference.

In [13]:
def run_walkforward(model_cls, model_kwargs, train, eval_series, seq_len, batch_size, epochs):
    common.set_seed()
    val_start = eval_series.index[0] - pd.Timedelta(days=VAL_DAYS)
    train_inner = train[train.index < val_start]
    val_inner = train[train.index >= val_start]

    scaler = common.SeriesScaler("minmax")
    train_scaled = scaler.fit_transform(train_inner).flatten()
    full_val_input = pd.concat([train_inner.tail(seq_len), val_inner])
    val_scaled = scaler.transform(full_val_input).flatten()
    full_eval_input = pd.concat([train.tail(seq_len), eval_series])
    eval_scaled = scaler.transform(full_eval_input).flatten()

    X_train, y_train = create_sequences(train_scaled, seq_len)
    X_val, y_val = create_sequences(val_scaled, seq_len)
    X_eval, y_eval = create_sequences(eval_scaled, seq_len)

    t0 = time.perf_counter()
    model = model_cls(seq_len, **model_kwargs)
    model.fit(X_train, y_train, X_val=X_val, y_val=y_val, batch_size=batch_size, epochs=epochs,
              early_stopping_patience=FINAL_EARLY_STOPPING_PATIENCE)
    fit_time = time.perf_counter() - t0

    t0 = time.perf_counter()
    preds_scaled = model.predict(X_eval)
    predict_time = time.perf_counter() - t0
    preds = scaler.inverse_transform(preds_scaled)
    return preds, fit_time, predict_time


eval_start = pd.Timestamp(EVAL_WEEK_START)
eval_end = pd.Timestamp(EVAL_WEEK_END) + pd.Timedelta(hours=23, minutes=50)
train_start = pd.Timestamp(TRAIN_START)

for square_id in meta["top3_squares"]:
    print(f"=== Square {square_id} ===")
    series = common.load_square_series(square_id)
    train = series[(series.index >= train_start) & (series.index < eval_start)]
    eval_series = series[(series.index >= eval_start) & (series.index <= eval_end)]
    print(f"train={len(train)} pts, eval={len(eval_series)} pts")

    preds, fit_t, pred_t = run_walkforward(
        LSTMModel, {"units": best_lstm["units"], "dropout": 0.2, "learning_rate": best_lstm["learning_rate"]},
        train, eval_series, best_lstm["sequence_length"], FINAL_BATCH_SIZE, FINAL_EPOCHS,
    )
    metrics = common.compute_metrics(eval_series.values, preds)
    print(f"LSTM sq={square_id} metrics={metrics} fit={fit_t:.1f}s predict={pred_t:.1f}s")
    result = {"square_id": square_id, "model": "LSTM", "predictions": preds, "eval_index": eval_series.index,
              "actual": eval_series.values, "metrics": metrics, "fit_time_s": fit_t, "predict_time_s": pred_t,
              "predict_time_per_step_ms": pred_t / len(eval_series) * 1000, "params": best_lstm}
    with open(f"results/pred_lstm_sq{square_id}.pkl", "wb") as f:
        pickle.dump(result, f)

    preds, fit_t, pred_t = run_walkforward(
        TCNModel, {"filters": best_tcn["filters"], "dropout": 0.2, "learning_rate": best_tcn["learning_rate"]},
        train, eval_series, best_tcn["sequence_length"], FINAL_BATCH_SIZE, FINAL_EPOCHS,
    )
    metrics = common.compute_metrics(eval_series.values, preds)
    print(f"TCN sq={square_id} metrics={metrics} fit={fit_t:.1f}s predict={pred_t:.1f}s")
    result = {"square_id": square_id, "model": "TCN", "predictions": preds, "eval_index": eval_series.index,
              "actual": eval_series.values, "metrics": metrics, "fit_time_s": fit_t, "predict_time_s": pred_t,
              "predict_time_per_step_ms": pred_t / len(eval_series) * 1000, "params": best_tcn}
    with open(f"results/pred_tcn_sq{square_id}.pkl", "wb") as f:
        pickle.dump(result, f)

print("Saved LSTM + TCN predictions/metrics for all 3 squares to results/pred_{lstm,tcn}_sq*.pkl")

=== Square 5161 ===
train=6480 pts, eval=1008 pts


/Users/manziivan453icloud.com/Downloads/Projects/IVAN/Formative1_Techinique1/venv/lib/python3.11/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


LSTM sq=5161 metrics={'MAE': 103.03702080966625, 'RMSE': 147.57547936808936, 'MAPE': 13.521135994682313} fit=19.8s predict=0.5s


TCN sq=5161 metrics={'MAE': 201.03778907816422, 'RMSE': 289.2019170116965, 'MAPE': 23.648698911512575} fit=37.4s predict=0.4s
=== Square 5059 ===
train=6480 pts, eval=1008 pts


/Users/manziivan453icloud.com/Downloads/Projects/IVAN/Formative1_Techinique1/venv/lib/python3.11/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


LSTM sq=5059 metrics={'MAE': 80.33501237375894, 'RMSE': 110.77604375559483, 'MAPE': 9.367346121844891} fit=22.8s predict=0.2s


TCN sq=5059 metrics={'MAE': 99.03876483733646, 'RMSE': 135.56465428242956, 'MAPE': 10.989538674479126} fit=39.0s predict=0.4s
=== Square 5259 ===
train=6480 pts, eval=1008 pts


/Users/manziivan453icloud.com/Downloads/Projects/IVAN/Formative1_Techinique1/venv/lib/python3.11/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


LSTM sq=5259 metrics={'MAE': 73.16044100797379, 'RMSE': 104.48978544721486, 'MAPE': 8.379797632189222} fit=25.7s predict=0.2s


TCN sq=5259 metrics={'MAE': 160.84843475672392, 'RMSE': 202.01652603254865, 'MAPE': 23.408123053391268} fit=81.1s predict=0.5s
Saved LSTM + TCN predictions/metrics for all 3 squares to results/pred_{lstm,tcn}_sq*.pkl
